# Task 2.1 — Dataset Selection

**Paper:** "Object Detection with Discriminatively Trained Part-Based Models"  
**Authors:** Felzenszwalb, Girshick, McAllester, Ramanan (2010)  
**Venue:** IEEE TPAMI

---

## Dataset Choice: Synthetic Part-Based Classification Dataset

We use `sklearn.datasets.make_classification` combined with a custom **part-based feature construction** to create a toy dataset that mirrors the conceptual structure of the DPM.

### Dataset Specification

| Property | Value |
|----------|-------|
| Number of samples | 500 (train: 400, test: 100) |
| Number of base features | 8 |
| Number of "part" feature groups | 4 groups of 2 features each |
| Classes | 2 (object vs. background) |
| Random seed | 42 |

### Feature Design

To simulate the DPM's structure, we construct features in a specific way:

1. **Root features (features 0–1):** Represent the global, coarse appearance of the object — analogous to the root filter's HOG response. These are informative features shared across all positive examples.

2. **Part features (features 2–7):** Divided into 3 groups of 2 features each, representing different "parts." Each group captures local appearance — analogous to part filter HOG responses.

3. **Spatial displacement features:** For each part, we add a "deformation" feature — a noisy offset from an expected value — simulating the displacement of parts from anchor positions. The model must learn to tolerate small deformations.

This construction allows us to:
- Train a **rigid baseline** (root features only) analogous to the Dalal & Triggs detector.
- Train a **part-based model** (root + part + deformation features) analogous to the full DPM.

## Why This Dataset Is Suitable

1. **Mirrors the DPM's core idea:** The dataset is constructed so that root features alone are somewhat informative, but adding part features significantly improves classification. This parallels how the DPM's root filter provides coarse detection while part filters add discriminative power.

2. **Deformation simulation:** By adding noisy displacement features, we simulate the part deformation model. The classifier must learn to weigh part appearance against deformation cost, just like the DPM.

3. **CPU-friendly:** 500 samples with ~14 features is trivially fast, allowing rapid experimentation and iteration.

4. **Controlled environment:** We know the ground truth feature structure, so we can precisely measure the effect of removing parts or deformation penalties in ablation studies.

## Limitations Compared to the Paper's Dataset

| Aspect | Paper (PASCAL VOC) | Our Toy Dataset |
|--------|-------------------|------------------|
| **Data type** | Real images with complex backgrounds | Synthetic tabular features |
| **Task** | Object detection (localization + classification) | Binary classification only |
| **Features** | HOG computed from pixel gradients | Directly generated numerical features |
| **Scale** | ~10K images, 20 categories | 500 samples, 2 classes |
| **Spatial structure** | True 2D spatial layout of parts | Features simulate but don't have real spatial arrangement |
| **Evaluation** | Average Precision (AP) with IoU overlap | Accuracy, F1-score |

The toy dataset captures the **conceptual structure** (root vs. parts vs. deformation) but cannot replicate the full complexity of real image-based object detection. This is a deliberate simplification for pedagogical clarity and CPU feasibility.

## Dataset Construction Code

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Step 1: Generate base features ---
# 8 informative features: 2 root + 6 part (3 parts x 2 features each)
X_base, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=8,
    n_redundant=0,
    n_clusters_per_class=2,
    class_sep=1.0,
    random_state=RANDOM_SEED
)

# --- Step 2: Label feature groups ---
root_features = X_base[:, 0:2]       # Root filter features (global appearance)
part1_features = X_base[:, 2:4]       # Part 1 features
part2_features = X_base[:, 4:6]       # Part 2 features
part3_features = X_base[:, 6:8]       # Part 3 features

# --- Step 3: Generate deformation features ---
# For each part, add a 2D displacement (dx, dy) from expected anchor
# Positives have small deformations; negatives have larger/random deformations
n_samples = X_base.shape[0]

def generate_deformation_features(y, n_parts=3, deform_std_pos=0.3, deform_std_neg=1.5):
    """Generate deformation features simulating part displacement."""
    deformations = np.zeros((len(y), n_parts * 2))  # dx, dy for each part
    for i in range(len(y)):
        if y[i] == 1:  # Positive: small deformations (parts near anchor)
            deformations[i] = np.random.normal(0, deform_std_pos, n_parts * 2)
        else:  # Negative: larger deformations (parts displaced randomly)
            deformations[i] = np.random.normal(0, deform_std_neg, n_parts * 2)
    return deformations

deformation_features = generate_deformation_features(y)

# --- Step 4: Combine all features ---
X_full = np.hstack([root_features, part1_features, part2_features, part3_features, deformation_features])

# Feature names for reference
feature_names = (
    ['root_f0', 'root_f1'] +
    ['part1_f0', 'part1_f1'] +
    ['part2_f0', 'part2_f1'] +
    ['part3_f0', 'part3_f1'] +
    ['part1_dx', 'part1_dy'] +
    ['part2_dx', 'part2_dy'] +
    ['part3_dx', 'part3_dy']
)

print(f"Full feature matrix shape: {X_full.shape}")
print(f"Feature names: {feature_names}")
print(f"Class distribution: {np.bincount(y)}")
print(f"  Class 0 (background): {np.sum(y==0)}")
print(f"  Class 1 (object): {np.sum(y==1)}")

**What this code does:**  
Generates a synthetic dataset with 500 samples and 14 features structured as root (2), parts (6), and deformation (6). This mirrors the DPM's feature decomposition (Section 3) where the detection score combines root filter response, part filter responses, and deformation costs.

The deformation features are designed so that positive examples have small displacements (parts near expected anchors) while negative examples have larger displacements, simulating the quadratic deformation penalty from Equation 3 of the paper.

In [ ]:
# --- Step 5: Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train class distribution: {np.bincount(y_train)}")
print(f"Test class distribution: {np.bincount(y_test)}")

In [ ]:
# --- Step 6: Visualize the dataset ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Root features
axes[0].scatter(X_full[y==0, 0], X_full[y==0, 1], c='blue', alpha=0.5, label='Background (0)', s=20)
axes[0].scatter(X_full[y==1, 0], X_full[y==1, 1], c='red', alpha=0.5, label='Object (1)', s=20)
axes[0].set_xlabel('root_f0')
axes[0].set_ylabel('root_f1')
axes[0].set_title('Root Features (Global Appearance)')
axes[0].legend()

# Plot 2: Part 1 features
axes[1].scatter(X_full[y==0, 2], X_full[y==0, 3], c='blue', alpha=0.5, label='Background (0)', s=20)
axes[1].scatter(X_full[y==1, 2], X_full[y==1, 3], c='red', alpha=0.5, label='Object (1)', s=20)
axes[1].set_xlabel('part1_f0')
axes[1].set_ylabel('part1_f1')
axes[1].set_title('Part 1 Features (Local Appearance)')
axes[1].legend()

# Plot 3: Deformation features for part 1
axes[2].scatter(X_full[y==0, 8], X_full[y==0, 9], c='blue', alpha=0.5, label='Background (0)', s=20)
axes[2].scatter(X_full[y==1, 8], X_full[y==1, 9], c='red', alpha=0.5, label='Object (1)', s=20)
axes[2].set_xlabel('part1_dx')
axes[2].set_ylabel('part1_dy')
axes[2].set_title('Part 1 Deformation (dx, dy)')
axes[2].legend()

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/dataset_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/dataset_visualization.png")

**What this code does:**  
Visualizes the dataset across three views: root features (global object appearance), part features (local appearance), and deformation features (spatial displacement). The plots show that root features alone provide some separability, but the deformation features (tighter cluster for positives) provide additional discriminative signal — motivating the full part-based model.

This parallels Section 3 of the paper, where detection score combines root response + part responses − deformation costs.

In [ ]:
# --- Step 7: Save dataset for other notebooks ---
os.makedirs('data', exist_ok=True)

df_full = pd.DataFrame(X_full, columns=feature_names)
df_full['label'] = y
df_full.to_csv('data/dpm_toy_dataset.csv', index=False)

# Save train/test split indices
train_indices = np.arange(len(y))[np.isin(np.arange(len(y)), 
    train_test_split(np.arange(len(y)), test_size=0.2, random_state=RANDOM_SEED, stratify=y)[0])]
test_indices = np.arange(len(y))[~np.isin(np.arange(len(y)), train_indices)]

np.save('data/train_indices.npy', train_indices)
np.save('data/test_indices.npy', test_indices)

print(f"Dataset saved to data/dpm_toy_dataset.csv")
print(f"Shape: {df_full.shape}")
print(f"\nFirst 5 rows:")
df_full.head()

**What this code does:**  
Saves the complete dataset and train/test split indices for use in subsequent notebooks (Task 2.2, 2.3, 3.1, 3.2). This ensures all experiments use identical data, supporting reproducibility.